In [ ]:
%%capture

import warnings
warnings.filterwarnings("ignore")

import altair as alt
import branca.colormap as cm
import folium
import gcsfs
import pandas as pd

import calitp_data_analysis.magics
import gt_extras as gte

from great_tables import GT, md

import chart_utils_for_operators as chart_utils
import prep_operator_data
import report_utils
import _color_palette
PREDICTIONS_GCS = "gs://calitp-analytics-data/data-analyses/rt_predictions/"

alt.data_transformers.enable("vegafusion")

one_month = pd.to_datetime("2026-02-01")
date_format = "%b %Y" # gtfs_digest/_new_operator_report_utils.py
one_month_formatted = one_month.strftime(date_format)

In [ ]:
operator_cols = ["day_type"]

schedule_cols = [
    "daily_trips", "daily_service_hours", "n_routes", "n_shapes",
    "n_stops", "num_stop_times", "daily_arrivals", 'n_days_schedule_and_rt'
]

tu_cols =['tu_messages_per_minute', 'pct_tu_trips', 'pct_tu_routes'] #'daily_tu_trips',

# split prediction cols into 2 separate tables so comparison is clearer
tu_prediction_cols1 = ["bus_catch_likelihood", "pct_tu_complete_minutes", "p50", "n_predictions"]
tu_prediction_cols2 = [
    "p25", "p75", "iqr",
    "prediction_padding_minutes", "avg_prediction_spread_minutes"
]

In [ ]:
equans = "Marin EQUANS Trip Update"
swiftly = "Marin Swiftly Trip Update"
mtc = "Bay Area 511 Marin TripUpdates"

df = pd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_operator_summary_marin.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("tu_name", "in", [equans, swiftly, mtc])]],
).pipe(prep_operator_data.merge_in_operator_percentiles)

In [ ]:
# Set variables for color bars used across maps, route dropdown, and great tables
PREDICTION_ERROR_COLORS =list(_color_palette.PREDICTION_ERROR_COLOR_PALETTE.values())
PREDICTION_ERROR_INDEX = [-5, -3, -1, 1, 3, 5]
PREDICTION_ERROR_LEGEND_CAPTION = "minutes (negative = late; positive = early)"

POS_BAR_COLOR = _color_palette.get_color("blueberry")
NEG_BAR_COLOR = _color_palette.get_color("vivid_cerise")

## Schedule + RT Summary Stats

In [ ]:
schedule_table = (
    GT(df[["tu_name"] + operator_cols + schedule_cols])
    .cols_label(
        daily_trips = "Daily Trips",
        daily_service_hours = "Daily Service Hours",
        n_routes = "# Routes",
        n_shapes = "# Shapes",
        n_stops = "# Stops",
        num_stop_times = "Total Scheduled Arrivals",
        daily_arrivals = "Daily Scheduled Arrivals",    
        n_days_schedule_and_rt = "# days with both RT",
    ).fmt_integer(
        columns = [
            "daily_trips", "n_routes", "n_shapes", "n_stops", 
            "num_stop_times", "daily_arrivals", "n_days_schedule_and_rt"]
    ).fmt_number(
        columns = ["daily_service_hours"], decimals=1
    ).tab_spanner(
        label="Schedule",
        columns = schedule_cols
    ).tab_header(
        title = "Schedule + RT Summary Metrics", 
        subtitle = f"{one_month_formatted}"
    )
)

chart_utils.format_great_table(schedule_table, day_type_grouping = False).tab_stub(
    groupname_col="tu_name", rowname_col = "day_type")

## General RT Metrics

<span style="color:#4477aa">**Update Availability Goal 1:** 2+ vehicle positions or trip updates messages per minute.</span>

<span style="color:#4477aa">**Update Availability Goal 2:** 100% routes are covered by RT, and 75%+ of trips have RT.</span>

In [ ]:
prediction_cols = tu_cols + tu_prediction_cols1 + tu_prediction_cols2

swiftly_df = (
    df[df.tu_name == swiftly]
    [operator_cols + prediction_cols]
    .rename(
        columns = {
            **{c: f'swiftly_{c}' for c in prediction_cols}
        })
)

equans_df = (
    df[df.tu_name == equans]
    [operator_cols + prediction_cols]
    .rename(
        columns = {
            **{c: f'equans_{c}' for c in prediction_cols}
        })
)

In [ ]:
df_wide = pd.merge(
    swiftly_df,
    equans_df,
    on = operator_cols,
    how = "inner"
)

In [ ]:
df_wide